# Model Registry, Versioning & Governance

Companion notebook for the [Model Registry lesson](https://ml-viz-ruby.vercel.app/courses/ml-in-practice/18-model-registry-and-governance).

**The idea in one sentence.** A model registry makes deployments **auditable and
reversible**: every version records its **lineage** (code, data, config), promotion goes
through a **gate** (beat production + clear a floor), and you can **roll back** instantly and
answer "which model served this decision, and how was it built?"

The essentials:

- **Registry + lineage:** each version stores its metric and the exact code/data/config that
  produced it — reproducibility and audit.
- **Promotion gate + rollback:** promote only a validated improvement; keep the previous
  version to roll back to.

We build a minimal registry, **validate the gate and the lineage audit**, then cover the
gotchas.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Dark style matching the site theme.
plt.style.use('dark_background')
plt.rcParams.update({
    'axes.edgecolor': '#475569',
    'axes.labelcolor': '#e2e8f0',
    'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8',
    'axes.titlecolor': '#e2e8f0',
    'figure.facecolor': '#0f1117',
    'axes.facecolor': '#1a1d27',
    'grid.color': '#2e3347',
    'savefig.facecolor': '#0f1117',
})
BRAND = '#6366f1'
TEAL = '#14b8a6'
ROSE = '#f43f5e'
YELLOW = '#eab308'

rng = np.random.default_rng(0)

## 1. A minimal registry

A registry is a versioned catalogue. Each version carries **lineage** (code commit, data version, config) and a **metric**, and lives in a **stage**: None → Staging → Production → Archived.

In [ ]:
class Registry:
    def __init__(self):
        self.versions = {}   # version -> record
        self.production = None
        self._n = 0
    def register(self, metric, code, data, config):
        self._n += 1
        v = f'v{self._n}'
        self.versions[v] = {'metric': metric, 'stage': 'Staging',
                            'lineage': {'code': code, 'data': data, 'config': config}}
        return v
    def promote(self, v):
        if self.production:
            self.versions[self.production]['stage'] = 'Archived'
        self.versions[v]['stage'] = 'Production'
        self.production = v
    def serving(self):
        return self.production

reg = Registry()
v1 = reg.register(0.81, 'abc123', 'data@2026-01', {'lr': 0.01, 'seed': 0})
reg.promote(v1)
print('serving:', reg.serving(), '->', reg.versions[v1]['lineage'])

## 2. Promotion gate + rollback

A new candidate is promoted only if it beats the current production model on a held-out metric and clears a floor. Rollback is a one-line stage change — re-point Production at the previous version.

In [ ]:
def gated_promote(reg, candidate_metric, floor, **lineage):
    prod_metric = reg.versions[reg.production]['metric'] if reg.production else -1
    v = reg.register(candidate_metric, **lineage)
    if candidate_metric > prod_metric and candidate_metric >= floor:
        reg.promote(v); print(f'{v} PROMOTED ({candidate_metric:.3f} > {prod_metric:.3f})')
    else:
        print(f'{v} rejected ({candidate_metric:.3f} vs prod {prod_metric:.3f}, floor {floor})')
    return v

v2 = gated_promote(reg, 0.85, 0.80, code='def456', data='data@2026-02', config={'lr':0.01,'seed':0})
v3 = gated_promote(reg, 0.79, 0.80, code='ghi789', data='data@2026-03', config={'lr':0.02,'seed':0})
print('serving now:', reg.serving())

# Rollback: a bad deploy of v2 -> re-point Production at v1
def rollback(reg, to_version):
    reg.versions[reg.production]['stage'] = 'Archived'
    reg.versions[to_version]['stage'] = 'Production'
    reg.production = to_version
rollback(reg, v1)
print('after rollback, serving:', reg.serving())

### Validate: the promotion gate ships only validated improvements

Governance means automation can't ship a regression. The gate promotes a candidate only if
it beats the current production metric *and* clears the quality floor. We register several
candidates and confirm only the qualifying one becomes production.

In [ ]:
reg_v = Registry()
gated_promote(reg_v, 0.80, floor=0.75, code='a', data='d1', config={'lr': 0.1})   # first: promoted
prod_after_first = reg_v.production
gated_promote(reg_v, 0.78, floor=0.75, code='b', data='d2', config={'lr': 0.1})   # worse: rejected
gated_promote(reg_v, 0.70, floor=0.75, code='c', data='d3', config={'lr': 0.1})   # below floor: rejected
gated_promote(reg_v, 0.85, floor=0.75, code='d', data='d4', config={'lr': 0.1})   # better: promoted
prod_metric = reg_v.versions[reg_v.production]['metric']
print(f'final production metric: {prod_metric:.2f}')
assert prod_metric == 0.85, 'only the qualifying (better + above-floor) candidate is in production'
print('\n✅ the promotion gate ships only a validated improvement — no regressions, no floor breaches')

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **no lineage** | you can't reproduce or audit a deployed model — a governance failure |
| **no promotion gate** | a regression or floor-breach ships automatically |
| **no rollback** | a bad deploy can't be reverted quickly; keep the prior version live |
| **metric-only gates** | a model can beat the metric yet fail fairness/latency checks — gate on those too |
| **stale registry** | the registry must reflect what's actually serving, or audits lie |

Demo: every version is reproducible and auditable from its stored lineage.

In [ ]:
# Auditability: every version keeps its LINEAGE, so you can answer 'which model made this
# decision and how was it built?' — essential for governance, debugging, and reproducibility.
# We confirm every registered version is fully reproducible from its stored lineage.
for v, rec in reg_v.versions.items():
    lin = rec['lineage']
    print(f"{v}: metric {rec['metric']:.2f}, code={lin['code']}, data={lin['data']}, config={lin['config']}")
    assert set(lin) >= {'code', 'data', 'config'}, 'every version records code+data+config lineage'
print('\nEvery version is reproducible + auditable from its lineage -> you can always answer')
print('"which model served this, and exactly how was it built?" (and roll back if needed).')

## ✏️ Your turn — the audit query

Implement `who_served(reg, version)` returning the lineage triplet (code, data, config) for a given version — the core of an audit trail ('which model decided this, trained on what?').

In [ ]:
def who_served(reg, version):
    """TODO(you): return reg.versions[version]['lineage']."""
    # TODO
    return ...


In [ ]:
lin = who_served(reg, v2)
print('v2 lineage:', lin)
assert lin['code'] == 'def456'
assert lin['data'] == 'data@2026-02'
assert lin['config']['lr'] == 0.01
print('✅ every version is reproducible and auditable from its lineage.')

<details>
<summary>Solution</summary>

```python
def who_served(reg, version):
    return reg.versions[version]['lineage']
```

Capture lineage **at registration time** — code commit, data snapshot, config/seed, metrics. Reconstructing it after an incident is painful or impossible.
</details>

## Recap

- A **registry** versions models across **stages**; deployment serves whatever is in Production.
- A **promotion gate** (beat prod + clear a floor) is where evaluation and sign-off attach.
- **Rollback** is a stage change, not a rebuild.
- **Lineage** (code, data, config, metrics) makes a version reproducible and auditable — capture it at registration.